# Chapter 17: QAE

---

**Prerequisites:**
- See `Chapter02_QuantumSoftware.ipynb` for installation instructions


In [3]:
# Setup and imports
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler 
from Chapter03_EngineeringOptimization_functions import (truss2x2,truss2x3,truss3x3)

from Chapter17_QAE_functions import build_grover_operator, myQAE

### QAE Concept

In [4]:
alpha = np.pi / 6
a_true = np.sin(alpha)**2   # 0.25

# Build circuit A
A = QuantumCircuit(1)
A.ry(2*alpha, 0)

# Run QAE with m = 3, 4, 5 precision qubits
for m in [3, 4, 5]:
    a_tilde, phi_tilde, counts = myQAE(A, [0], m, nShots=10000)
    print(f"m={m}: phi={phi_tilde:.4f}  a_tilde={a_tilde:.4f}  "
          f"error={abs(a_tilde - a_true):.4f}")

AerError: 'unknown instruction: circuit-47'

###  QAE Accuracy

In [5]:
a_values = [0.1, 0.25, 0.5, 0.75, 0.9]
m = 4

print(f"{'a_true':>8} {'a_tilde':>8} {'error':>8}")
for a in a_values:
    alpha = np.arcsin(np.sqrt(a))
    A = QuantumCircuit(1)
    A.ry(2*alpha, 0)
    a_tilde, _, _ = myQAE(A, [0], m, nShots=10000)
    print(f"{a:>8.3f} {a_tilde:>8.4f} {abs(a_tilde-a):>8.4f}")


  a_true  a_tilde    error


AerError: 'unknown instruction: circuit-101'

### QAE vs. Classical

In [ ]:
# Classical result
f = np.array([1., 0., 0.5, 0.2])
f = f / np.linalg.norm(f)
A_mat = np.array([[ 1.,  0.,  0., -0.5],
                  [ 0.,  1.,  0.,  0.  ],
                  [ 0.,  0.,  1.,  0.  ],
                  [-0.5, 0.,  0.,  1.  ]])
x_vec = np.array([0.6, 0.8, 0., 0.])
classical = np.abs(f @ A_mat @ x_vec)
print(f"Classical: {classical:.4f}")

# Build observable circuit A_obs (LCU + U_f^dag)
A_obs = build_observable_circuit(A_mat, x_vec, f)  # see code repo

# Monte Carlo: sample A_obs and count all-zeros outcomes
mc_estimate = monte_carlo_observable(A_obs, n_shots=10000)
print(f"Monte Carlo (10,000 shots): {mc_estimate:.4f}")

# QAE: m=5 precision qubits
a_tilde, _, _ = myQAE(A_obs, good_qubits=[0,1,2], m=5, nShots=1000)
alpha_lcu = compute_alpha(A_mat)
p_success  = np.linalg.norm(A_mat @ x_vec)**2 / alpha_lcu**2
qae_estimate = np.sqrt(a_tilde) * alpha_lcu * np.sqrt(p_success)
print(f"QAE (m=5, ~1000 circuit calls): {qae_estimate:.4f}")